# 08 — Data Scaling
**Spacecraft Telemetry Anomaly Detection | Stage 1**

---
**Goal:** Normalise all 50 parameters onto comparable scales before feeding them to ML models.

> Without scaling, a parameter like RF_SIGNAL_STRENGTH (range: −85 to −60 dBm) would  
> numerically dominate a parameter like GYRO_X (range: −0.1 to +0.1 rad/s) — even though  
> both are equally important for anomaly detection.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler, MinMaxScaler

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.facecolor':'white', 'axes.facecolor':'#f8f9fa',
                     'axes.titlesize':11, 'font.size':10})
os.makedirs('plots_v2', exist_ok=True)
os.makedirs('processed_v2', exist_ok=True)

PALETTE = ['#1f77b4','#2ca02c','#d62728','#9467bd','#8c564b']

# Load the wide-format dataset produced in Section 07
tel_wide   = pd.read_csv('processed_v2/telemetry_wide.csv', parse_dates=['timestamp'])
param_cols = [c for c in tel_wide.columns if c != 'timestamp']
X_raw      = tel_wide[param_cols].values  # Convert to numpy array for sklearn

print('Wide format loaded:', tel_wide.shape)
print('Parameter columns  :', len(param_cols))
print('Raw value ranges   :')
print(f'  Global min: {X_raw.min():.2f}  |  Global max: {X_raw.max():.2f}')
print('  --> Huge range -- scaling required')

### 8.1 StandardScaler — Zero Mean, Unit Variance
Formula: `z = (x − mean) / std`

In [ ]:
# CRITICAL: fit_transform() here learns mean and std from THIS training set
# In Stage 2 (model evaluation), we will ONLY use transform() on test/anomaly data
# Never refit the scaler on test data -- that would cause data leakage
std_scaler = StandardScaler()
X_std      = std_scaler.fit_transform(X_raw)
df_std     = pd.DataFrame(X_std, columns=param_cols)

print('StandardScaler applied — per-column verification:')
print(f'  Mean range : [{df_std.mean().min():.6f}, {df_std.mean().max():.6f}]  (should be ~0)')
print(f'  Std range  : [{df_std.std().min():.6f}, {df_std.std().max():.6f}]  (should be ~1)')

# RESULT: Mean ≈ 0 and Std ≈ 1 for every parameter
# All 50 parameters are now on the same scale -- no parameter dominates
display(df_std.describe().round(3))

### 8.2 MinMaxScaler — Compress to [0, 1] Range
Formula: `x' = (x − min) / (max − min)`

In [ ]:
mm_scaler = MinMaxScaler()  # Scales each column to exactly [0, 1]
X_mm      = mm_scaler.fit_transform(X_raw)
df_mm     = pd.DataFrame(X_mm, columns=param_cols)

print('MinMaxScaler applied — per-column verification:')
print(f'  Min range  : [{df_mm.min().min():.4f}, {df_mm.min().max():.4f}]  (should be 0.0)')
print(f'  Max range  : [{df_mm.max().min():.4f}, {df_mm.max().max():.4f}]  (should be 1.0)')

# RESULT: Every parameter's minimum maps to 0.0 and maximum maps to 1.0
# Useful for autoencoder output layers with sigmoid activation (output range 0-1)
display(df_mm.describe().round(3))

### 8.3 Visual Comparison — Raw vs Scaled
Same 5 parameters, three scaling states side by side.

In [ ]:
SEL = ['BATT_VOLTAGE_1','SOLAR_POWER_TOTAL','OBC_TEMP',
       'RF_SIGNAL_STRENGTH','GYRO_X']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Scaling Comparison — 5 Parameters (same data, different scales)',
             fontsize=12, fontweight='bold')

for ax, data, title in zip(axes,
                            [tel_wide[SEL], df_std[SEL], df_mm[SEL]],
                            ['Raw Values (different units — unscalable)',
                             'StandardScaler  (mean=0, std=1)',
                             'MinMaxScaler  [0, 1]']):
    for col, c in zip(SEL, PALETTE):
        ax.hist(data[col], bins=25, alpha=0.55, label=col.replace('_',' '), color=c)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=6)

plt.tight_layout()
plt.savefig('plots_v2/08_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

# KEY RESULTS:
# - Raw: all 5 histograms on different x-axis positions -- GYRO_X is invisible next to RF_SIGNAL
# - StandardScaler: all 5 distributions centred at 0 -- now truly comparable
# - MinMaxScaler: all 5 distributions mapped to [0,1] -- all overlap on same x-axis
# --> After scaling, the SHAPE of each distribution is preserved (Gaussian stays Gaussian)
#     but the spread/centre is standardised

### 8.4 Raw vs Scaled Range — All 50 Parameters

In [ ]:
raw_df = tel_wide[param_cols]
ranges = pd.DataFrame({
    'Raw Min':    raw_df.min().round(2),
    'Raw Max':    raw_df.max().round(2),
    'Std Min':    df_std.min().round(3),
    'Std Max':    df_std.max().round(3),
    'MM Min':     df_mm.min().round(3),
    'MM Max':     df_mm.max().round(3),
})
display(ranges)

# RESULT: Raw min/max vary wildly (-100 to +1000+ depending on parameter)
# After Standard scaling: all Std Max values are in similar single-digit range
# After MinMax scaling: all MM Min = 0.0 and all MM Max = 1.0 exactly

### 8.5 Which Scaler to Use — Decision Guide

In [ ]:
scaler_table = pd.DataFrame({
    'Property':       ['Formula','Output range','Preserves shape',
                       'Sensitive to outliers','Best model fit',
                       'After anomaly injection'],
    'StandardScaler': [
        'z = (x − mean) / std',
        'Unbounded, centred at 0',
        'Yes — Gaussian stays Gaussian',
        'Yes — one outlier shifts mean/std',
        'Isolation Forest, One-Class SVM, PCA',
        'Anomaly points will have large |z| → model flags them',
    ],
    'MinMaxScaler': [
        "x' = (x − min) / (max − min)",
        '[0, 1]',
        'Yes',
        'Yes — one extreme outlier compresses all inliers into tiny band',
        'GRU / TCN Autoencoder (sigmoid/tanh activations)',
        'Anomalies may push values slightly outside [0,1] → detectable',
    ],
})
display(scaler_table)

In [ ]:
# Save both scaled datasets
df_std_out = df_std.copy()
df_std_out.insert(0, 'timestamp', tel_wide['timestamp'].values)
df_std_out.to_csv('processed_v2/telemetry_standard_scaled.csv', index=False)

df_mm_out = df_mm.copy()
df_mm_out.insert(0, 'timestamp', tel_wide['timestamp'].values)
df_mm_out.to_csv('processed_v2/telemetry_minmax_scaled.csv', index=False)

print('Saved: processed_v2/telemetry_standard_scaled.csv  --> for Isolation Forest / OCSVM')
print('Saved: processed_v2/telemetry_minmax_scaled.csv    --> for GRU / TCN Autoencoder')
print('\nBoth files are Stage-1 complete and ready for model input in Stage 2.')